# 프로젝트 3. 부동산 데이터 분석

- 작성: 양재일 (서울사이버대학교 인공지능학과)
- 역할: 공공 실거래 집계를 불러와 지역별 가격·거래량을 비교하고, 공인중개·담보 실무 관점으로 해석한다.
- 사용 도구: Python, pandas, matplotlib
- 원자료: 국토교통부 아파트·연립다세대·오피스텔 **매매 실거래가** (공공데이터포털)
- 사용 파일: 서울 예산 지도(DAS Lab)가 구·월 단위로 집계한 CSV (CC BY 4.0)
- 기준월: 파일의 `as_of` 열 (현재 공개본은 **2026-09**)

> 투자·매수·대출 권유가 아닙니다. 학습용 집계이며, 개별 단지의 시세와는 다를 수 있습니다.

## 1. 문제 정의

공인중개사와 여신 실무에서 반복해서 확인하던 질문은 두 가지입니다.

1. **이 지역 아파트의 거래 가격 수준은 어디쯤인가?** (담보·시세의 기준)
2. **실제로 거래가 되고 있는가?** (가격만 높고 거래가 없으면 유동성이 떨어진다)

국토교통부 실거래가 원본은 건수가 많아 API 키가 필요합니다. 이 노트북은 같은 원자료를 구 단위로 집계한 **공개 CSV**를 써서, 키 없이 서울 25개 자치구의 가격과 거래 추세를 봅니다.

분석 범위는 **아파트 · 전용 84㎡ 구간**입니다. 실무에서 가장 많이 비교하는 국민평형에 가깝습니다.

## 2. 데이터와 출처

| 구분 | 내용 |
| --- | --- |
| 원자료 | 국토교통부 실거래가 공개시스템 / 공공데이터포털 |
| 집계 파일 | [서울 동네별 실거래가 집계](https://seoul-heatmap.daslab.co.kr/open-data/) |
| 라이선스 | CC BY 4.0 (출처 표기) |
| 출처 표기 | 서울 예산 지도(DAS Lab), “서울 동네별 실거래가 집계”, 국토교통부 실거래가 기준 2026년 9월, https://seoul-heatmap.daslab.co.kr/open-data/ (CC BY 4.0) |

가격은 월별 중위 거래가를 거래 건수로 가중 평균한 값입니다. 최근 달이 비어 있으면 신고가 덜 들어온 달일 수 있습니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib import font_manager as fm
import urllib.request

def setup_korean_font():
    dest = Path("NanumGothicCoding.ttf")
    candidates = [
        dest,
        Path("/usr/share/fonts/truetype/nanum/NanumGothic.ttf"),
        Path("C:/Windows/Fonts/malgun.ttf"),
    ]
    font_path = next((p for p in candidates if p.exists()), None)
    if font_path is None:
        url = "https://github.com/google/fonts/raw/main/ofl/nanumgothiccoding/NanumGothicCoding-Regular.ttf"
        urllib.request.urlretrieve(url, dest)
        font_path = dest
    fm.fontManager.addfont(str(font_path))
    name = fm.FontProperties(fname=str(font_path)).get_name()
    plt.rcParams["font.family"] = name
    plt.rcParams["font.sans-serif"] = [name]
    plt.rcParams["axes.unicode_minus"] = False
    print("한글 글꼴 적용:", name)

setup_korean_font()
plt.rcParams["figure.figsize"] = (8, 4.5)
print("라이브러리 준비 완료")

In [ ]:
BASE = "https://seoul-heatmap.daslab.co.kr/open-data"
gu_now = pd.read_csv(f"{BASE}/gu-current.csv", encoding="utf-8-sig")
gu_month = pd.read_csv(f"{BASE}/gu-monthly.csv", encoding="utf-8-sig")

print("구별 현재가 행:", len(gu_now), "/ 기준월:", gu_now["as_of"].iloc[0])
print("구별 월별 행:", len(gu_month), "/ 기간:", gu_month["month"].min(), "~", gu_month["month"].max())

apt84 = gu_now[(gu_now["property_type"] == "apt") & (gu_now["area_key"].astype(str) == "84")].copy()
apt84_m = gu_month[(gu_month["property_type"] == "apt") & (gu_month["area_key"].astype(str) == "84")].copy()
apt84_m["month"] = pd.to_datetime(apt84_m["month"] + "-01")

print("아파트 전용 84㎡ 자치구 수:", apt84["gu"].nunique())
apt84[["gu", "price_eok", "trades", "window_months"]].sort_values("price_eok", ascending=False).head()

## 3. 방법

1. 아파트 전용 84㎡만 남긴다.
2. 자치구별 **현재 가격(억원)** 과 **거래 건수**를 막대그래프로 비교한다.
3. 최근 24개월 중위가·거래 건수 추세를 강남 3구와 그 외 대표 구로 본다.
4. 자치구 중심 좌표에 가격을 색, 거래 건수를 크기로 올려 **지도형 산점도**를 그린다.

In [ ]:
rank = apt84.sort_values("price_eok")
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
axes[0].barh(rank["gu"], rank["price_eok"], color="#bd5d38")
axes[0].set_xlabel("현재 가격 (억원)")
axes[0].set_title(f"서울 자치구 아파트 전용 84㎡ 가격\n기준 {apt84['as_of'].iloc[0]}")
axes[1].barh(rank["gu"], rank["trades"], color="#4c9a8a")
axes[1].set_xlabel("집계에 쓴 거래 건수")
axes[1].set_title("같은 구의 거래 건수")
plt.tight_layout()
plt.show()

print("가격 상위 5")
display(rank.tail(5)[["gu", "price_eok", "trades"]][::-1])
print("가격 하위 5")
display(rank.head(5)[["gu", "price_eok", "trades"]])

In [ ]:
watch = ["강남구", "서초구", "송파구", "노원구", "도봉구", "강서구"]
trend = apt84_m[apt84_m["gu"].isin(watch)]

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
for gu, part in trend.groupby("gu"):
    part = part.sort_values("month")
    axes[0].plot(part["month"], part["median_price_man"] / 10000, marker="o", label=gu)
    axes[1].plot(part["month"], part["trades"], marker="o", label=gu)
axes[0].set_ylabel("중위 거래가 (억원)")
axes[0].set_title("자치구별 아파트 전용 84㎡ 월별 중위가")
axes[0].legend(ncol=3, loc="upper left")
axes[1].set_ylabel("거래 건수")
axes[1].set_title("같은 구의 월별 거래 건수")
axes[1].legend(ncol=3, loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
GU_XY = {
    "종로구": (126.9790, 37.5735), "중구": (126.9979, 37.5641), "용산구": (126.9900, 37.5326),
    "성동구": (127.0369, 37.5634), "광진구": (127.0823, 37.5384), "동대문구": (127.0396, 37.5744),
    "중랑구": (127.0926, 37.6063), "성북구": (127.0167, 37.5894), "강북구": (127.0257, 37.6396),
    "도봉구": (127.0471, 37.6688), "노원구": (127.0568, 37.6542), "은평구": (126.9291, 37.6027),
    "서대문구": (126.9368, 37.5791), "마포구": (126.9016, 37.5663), "양천구": (126.8666, 37.5170),
    "강서구": (126.8495, 37.5509), "구로구": (126.8874, 37.4954), "금천구": (126.9020, 37.4519),
    "영등포구": (126.8962, 37.5264), "동작구": (126.9393, 37.5124), "관악구": (126.9516, 37.4784),
    "서초구": (127.0324, 37.4837), "강남구": (127.0473, 37.5172), "송파구": (127.1059, 37.5145),
    "강동구": (127.1238, 37.5301),
}
geo = apt84.copy()
geo["경도"] = geo["gu"].map(lambda g: GU_XY[g][0])
geo["위도"] = geo["gu"].map(lambda g: GU_XY[g][1])

fig, ax = plt.subplots(figsize=(8.5, 8))
sc = ax.scatter(
    geo["경도"], geo["위도"],
    c=geo["price_eok"], s=geo["trades"] / 2,
    cmap="YlOrRd", alpha=0.9, edgecolors="#343a40", linewidths=0.4,
)
for _, row in geo.iterrows():
    ax.text(row["경도"], row["위도"] + 0.006, row["gu"].replace("구", ""), ha="center", fontsize=8)
cb = plt.colorbar(sc, ax=ax, shrink=0.8)
cb.set_label("아파트 전용 84㎡ 가격 (억원)")
ax.set_title("서울 자치구 위치 · 색=가격 · 크기=거래 건수")
ax.set_xlabel("경도")
ax.set_ylabel("위도")
ax.set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()

## 4. 결과 해석 (중개·담보 실무 관점)

숫자는 기준월과 신고 시점에 따라 달라집니다. 읽기 포인트는 다음과 같습니다.

1. **한강 이남 핵심 구(강남·서초·송파)** 의 전용 84㎡ 가격이 뚜렷이 높다. 담보 한도를 잡을 때 같은 ‘서울 아파트’라도 구별로 기준이 갈린다.
2. **가격이 높은 구가 거래도 많은 것은 아니다.** 노원·도봉처럼 가격은 낮아도 거래 건수가 많은 곳은 환금성이 상대적으로 낫다.
3. 월별 추세에서 가격이 올라도 거래 건수가 줄면, 실무에서는 호가만 오르고 실제 체결은 줄어든 구간으로 본다. 담보 평가를 최근 소수 거래에만 의존하면 왜곡된다.
4. 지도형 산점도는 강남 3구가 동남쪽에 모이고, 도봉·노원·강북이 북쪽에 모여 가격 색이 옅어지는 패턴을 한눈에 보여 준다.

공인중개 현장에서는 이 구 단위 숫자를 **초깃값**으로만 쓰고, 단지·층·향·리모델링 여부를 반드시 덧붙인다.

## 5. 배운 점과 한계

**배운 점**
- 공공 실거래 원본을 구·면적 구간으로 집계하면, API 키 없이도 지역 비교가 가능하다.
- 가격과 거래 건수를 나란히 봐야 ‘비싼 곳’과 ‘잘 팔리는 곳’을 구분할 수 있다.
- 출처·기준월·면적 구간을 같이 적어야 그래프를 오해하지 않는다.

**한계**
- 개별 단지·층수가 없다. 강남구 안에서도 시세 차이가 크다.
- 최근 달은 신고 지연으로 거래 건수가 적게 나올 수 있다.
- 전세·월세는 이 파일에 없다.

**다음 개선**
- 법정동 단위 CSV로 재개발·역세권 동을 따로 본다.
- folium 행정구 경계로 단계구분도를 그린다.
- 프로젝트 1(신용)과 연결해, 담보 지역 위험과 차주 신용을 같이 보는 구조를 만든다.